In [ ]:
import os

In [ ]:
print(os.getcwd())
os.chdir('../')
print(os.getcwd())

In [ ]:
from pdf_markdown_converter import PdfToMarkdownConverter

converter = PdfToMarkdownConverter(
    input_dir="data/PDF",
    output_dir="data/markdown",
)

summary = converter.convert_directory()

print(summary.total_files)
print(summary.converted_files)
print(summary.failed_files)


***

In [ ]:
import os 

print(os.getcwd())
os.chdir('../')
print(os.getcwd())

In [ ]:
from pathlib import Path

from markdown_vector_indexer import (
    ChunkingConfig,
    IndexingConfig,
    MarkdownVectorIndexer,
    OpenAIEmbeddingConfig,
)

config = IndexingConfig(
    input_dir=Path("data/markdown"),
    chroma_dir=Path("data/chromadb"),
    keywords_output_dir=Path("data/keywords"),
    collection_name="markdown_documents",
    reset_collection=True,
    chunking=ChunkingConfig(
        max_characters=1200,
        overlap_characters=200,
        min_characters=150,
    ),
    embedding=OpenAIEmbeddingConfig(
        model="text-embedding-3-small",
        batch_size=32,
    ),
)

indexer = MarkdownVectorIndexer(config)
summary = indexer.index_documents()

print(summary)

In [ ]:
import chromadb
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

openai_client = OpenAI()
chroma_client = chromadb.PersistentClient(path="data/chromadb")
collection = chroma_client.get_collection("markdown_documents")


def buscar_contexto(pergunta: str, n_results: int = 3) -> list[dict]:
    embedding = openai_client.embeddings.create(
        model="text-embedding-3-small",
        input=pergunta,
        encoding_format="float",
    ).data[0].embedding

    resultados = collection.query(
        query_embeddings=[embedding],
        n_results=n_results,
    )

    itens = []
    for i in range(len(resultados["ids"][0])):
        itens.append(
            {
                "id": resultados["ids"][0][i],
                "document": resultados["documents"][0][i],
                "metadata": resultados["metadatas"][0][i],
                "distance": resultados["distances"][0][i],
            }
        )
    return itens


In [ ]:
hits = buscar_contexto("Quando o rastreamento deve ser anual?")

for hit in hits:

    print(hit["metadata"]["relative_path"], hit["distance"])
    print(hit["document"][:300])

    print()
    print('----'*20)
    print()

In [ ]:
hits = buscar_contexto("Conjunto de regras para prescriçnao de medicamentos")

for hit in hits:

    print(hit["metadata"]["relative_path"], hit["distance"])
    print(hit["document"][:300])

    print()
    print('----'*20)
    print()

***

In [ ]:
import os 

print(os.getcwd())
os.chdir('../')
print(os.getcwd())

from dotenv import load_dotenv
from markdown_vector_indexer import DecisionFlowPipeline

In [ ]:
load_dotenv()
pipeline = DecisionFlowPipeline()

In [ ]:
print('LLM:', pipeline.generation.response_model)

In [ ]:
result = pipeline.query(
    "Criar árvore de decisão com enfoque no suporte à decisão médica para recomendação de medicamentos a serem prescritos ao paciente."
    "Considere que, na prática, a árvore devere permitir conclusões (i.e, recomendação de medicamentos) tomando como ponto de partida:"
    "1. o histórico de diagnósticos CID-10 do paciente;"
    "2. os medicamentos atualmente que o paciente usa;"
    "3. histórico de comorbidades do paciente;"
    "4. parâmetros antropométricos do paciente (peso, altura, IMC)."
)

In [ ]:
result

In [ ]:
result.flows

In [ ]:
for flow in result.flows:
    print("FLOW-ID:", flow.flowID)
    print("NODE-ID:", flow.nodeID)
    print("REGRA:", flow.regra)
    print("PRÓXIMO PASSO:", flow.proximo_passo)
    print("SAÍDA ALTERNATIVA:", flow.saida_alternativa)
    print("-" * 100)


---

In [ ]:
ruleset = {
    "flow_id": "F1",
    "start_node": "N1",
    "nodes": {
        "N1": {
            "type": "decision",
            "condition": 'context.get("cid10") in {"E10", "E11", "E13", "E14"}',
            "if_true": "N2",
            "if_false": "N3",
            "action_if_true": None,
            "action_if_false": None,
            "description": "Diagnóstico CID-10 compatível com diabetes mellitus."
        },

        "N2": {
            "type": "decision",
            "condition": (
                '(context.get("glicemia", 0) >= 126) or '
                '(context.get("hemoglobina_glicada", 0) >= 6.5) or '
                '(context.get("teste_tolerancia_oral_glicose", 0) >= 200)'
            ),
            "if_true": "N4",
            "if_false": "N5",
            "action_if_true": None,
            "action_if_false": None,
            "description": "Critérios laboratoriais para diabetes."
        },

        "N4": {
            "type": "decision",
            "condition": (
                '(context.get("imc", 0) >= 40) or '
                'context.get("antecedente_familiar_primeiro_grau_diabetes_mellitus", False) or '
                'context.get("sedentarismo", False) or '
                'context.get("hipertensao", False) or '
                'context.get("doenca_arterial_coronariana", False) or '
                '(context.get("triglicerides", 0) >= 250) or '
                '(context.get("hdl_c", 9999) < 35) or '
                'context.get("sindrome_ovarios_policisticos", False)'
            ),
            "if_true": "N6",
            "if_false": "N7",
            "action_if_true": None,
            "action_if_false": None,
            "description": "Fatores de risco cardiometabólicos."
        },

        "N6": {
            "type": "decision",
            "condition": 'context.get("estratificacao_risco") == "muito_alta"',
            "if_true": "N8",
            "if_false": "FIM",
            "action_if_true": {
                "type": "prescription",
                "description": "Prescrever ácido acetil salicílico 100mg via oral 1x ao dia",
                "medications": [
                    {
                        "name": "ácido acetil salicílico",
                        "dose": "100mg",
                        "route": "via oral",
                        "frequency": "1x ao dia"
                    }
                ]
            },
            "action_if_false": None,
            "description": "Prescrição de AAS em risco muito alto."
        },

        "N8": {
            "type": "decision",
            "condition": 'context.get("estratificacao_risco") in {"alta", "muito_alta"}',
            "if_true": "N9",
            "if_false": "FIM",
            "action_if_true": {
                "type": "prescription",
                "description": "Prescrever estatina de alta potência",
                "medications": [
                    {
                        "name": "Atorvastatina",
                        "dose": "40mg",
                        "route": "via oral",
                        "frequency": "1x ao dia"
                    },
                    {
                        "name": "Rosuvastatina",
                        "dose": "20mg",
                        "route": "via oral",
                        "frequency": "1x ao dia"
                    }
                ],
                "selection_mode": "choose_one"
            },
            "action_if_false": None,
            "description": "Prescrição de estatina de alta potência para risco alto ou muito alto."
        },

        "N9": {
            "type": "decision",
            "condition": 'context.get("hipertensao_arterial", False)',
            "if_true": "N10",
            "if_false": "FIM",
            "action_if_true": {
                "type": "prescription",
                "description": "Prescrever anti-hipertensivo",
                "medications": [
                    {
                        "name": "Losartana",
                        "dose": "50mg",
                        "route": "via oral",
                        "frequency": "1x ao dia"
                    },
                    {
                        "name": "Enalapril",
                        "dose": "10mg",
                        "route": "via oral",
                        "frequency": "1x ao dia"
                    }
                ],
                "selection_mode": "choose_one"
            },
            "action_if_false": None,
            "description": "Prescrição anti-hipertensiva."
        },

        "N10": {
            "type": "decision",
            "condition": (
                '(context.get("taxa_filtracao_glomerular", 9999) < 60) or '
                '(context.get("albuminuria_mg_por_g_creatinina", 0) > 30)'
            ),
            "if_true": "N11",
            "if_false": "FIM",
            "action_if_true": {
                "type": "prescription",
                "description": "Prescrever tratamento para doença renal diabética",
                "medications": [
                    {
                        "name": "Losartana",
                        "dose": "50mg",
                        "route": "via oral",
                        "frequency": "1x ao dia"
                    },
                    {
                        "name": "Valsartana",
                        "dose": "80mg",
                        "route": "via oral",
                        "frequency": "1x ao dia"
                    },
                    {
                        "name": "Enalapril",
                        "dose": "10mg",
                        "route": "via oral",
                        "frequency": "1x ao dia"
                    },
                    {
                        "name": "Lisinopril",
                        "dose": "10mg",
                        "route": "via oral",
                        "frequency": "1x ao dia"
                    },
                    {
                        "name": "Empagliflozina",
                        "dose": "10mg",
                        "route": "via oral",
                        "frequency": "1x ao dia"
                    },
                    {
                        "name": "Dapaglifozina",
                        "dose": "10mg",
                        "route": "via oral",
                        "frequency": "1x ao dia"
                    }
                ],
                "selection_mode": "choose_one_or_more"
            },
            "action_if_false": None,
            "description": "Prescrição para doença renal diabética."
        },

        "N11": {
            "type": "decision",
            "condition": (
                'context.get("prediabetes", False) and '
                '(context.get("idade", 9999) < 60) and ('
                '(context.get("imc", 0) >= 35) or '
                '(context.get("glicemia_jejum", 0) >= 110) or '
                '(context.get("hba1c", 0) >= 6.0) or '
                'context.get("historico_diabetes_mellitus_gestacional_cid10_o24_4", False)'
                ')'
            ),
            "if_true": "FIM",
            "if_false": "FIM",
            "action_if_true": {
                "type": "prescription",
                "description": "Prescrever Metformina 500mg XR via oral 1x ao dia",
                "medications": [
                    {
                        "name": "Metformina XR",
                        "dose": "500mg",
                        "route": "via oral",
                        "frequency": "1x ao dia"
                    }
                ]
            },
            "action_if_false": None,
            "description": "Prescrição de metformina em prediabetes sob critérios adicionais."
        },

        "N3": {
            "type": "terminal",
            "result": "Fluxo encerrado em N3",
            "description": "Nó citado no fluxo, mas sem detalhamento adicional."
        },

        "N5": {
            "type": "terminal",
            "result": "Fluxo encerrado em N5",
            "description": "Nó citado no fluxo, mas sem detalhamento adicional."
        },

        "N7": {
            "type": "terminal",
            "result": "Fluxo encerrado em N7",
            "description": "Nó citado no fluxo, mas sem detalhamento adicional."
        },

        "FIM": {
            "type": "terminal",
            "result": "Fluxo encerrado",
            "description": "Fim do fluxo."
        }
    }
}

In [ ]:
from typing import Any, Dict, List, Optional


def evaluate_condition(condition: str, context: Dict[str, Any]) -> bool:
    """
    Avalia uma expressão booleana Python armazenada como string.

    A expressão deve usar apenas `context.get(...)` para acessar variáveis.
    Exemplo:
        'context.get("glicemia", 0) >= 126'

    Parâmetros
    ----------
    condition : str
        Expressão booleana em formato string.
    context : dict
        Dicionário com os dados de entrada do paciente/contexto.

    Retorno
    -------
    bool
        Resultado da avaliação da condição.

    Observação
    ----------
    Esta implementação usa eval com builtins desabilitados para manter
    a interface simples e compatível com o formato de ruleset adotado.
    Em produção, o ideal é substituir por um avaliador seguro via AST/parser.
    """
    if not isinstance(condition, str) or not condition.strip():
        raise ValueError("A condição deve ser uma string não vazia.")

    safe_globals = {"__builtins__": {}}
    safe_locals = {"context": context}

    result = eval(condition, safe_globals, safe_locals)

    if not isinstance(result, bool):
        raise ValueError(f"A condição não retornou booleano: {condition}")

    return result


def execute_ruleset(
    ruleset: Dict[str, Any],
    context: Dict[str, Any],
    max_steps: int = 100
) -> Dict[str, Any]:
    """
    Executa um ruleset baseado em nós de decisão.

    Parâmetros
    ----------
    ruleset : dict
        Dicionário contendo:
            - flow_id
            - start_node
            - nodes
    context : dict
        Dados de entrada usados para resolver as condições.
    max_steps : int
        Limite de segurança para evitar loops infinitos.

    Retorno
    -------
    dict
        Estrutura com:
            - flow_id
            - final_node
            - status
            - trace
            - actions
            - warnings

    Formato do retorno
    ------------------
    {
        "flow_id": "F1",
        "final_node": "FIM",
        "status": "completed",
        "trace": [...],
        "actions": [...],
        "warnings": [...]
    }
    """
    if not isinstance(ruleset, dict):
        raise ValueError("`ruleset` deve ser um dicionário.")

    flow_id = ruleset.get("flow_id")
    start_node = ruleset.get("start_node")
    nodes = ruleset.get("nodes")

    if not flow_id:
        raise ValueError("O ruleset deve conter `flow_id`.")
    if not start_node:
        raise ValueError("O ruleset deve conter `start_node`.")
    if not isinstance(nodes, dict) or not nodes:
        raise ValueError("O ruleset deve conter `nodes` como dicionário não vazio.")

    current_node_id = start_node
    trace: List[Dict[str, Any]] = []
    actions: List[Dict[str, Any]] = []
    warnings: List[str] = []

    for step_index in range(max_steps):
        node = nodes.get(current_node_id)

        if node is None:
            raise KeyError(f"Nó `{current_node_id}` não encontrado no ruleset.")

        node_type = node.get("type")

        if node_type == "terminal":
            trace.append({
                "step": step_index + 1,
                "node_id": current_node_id,
                "node_type": "terminal",
                "result": node.get("result"),
                "description": node.get("description")
            })

            return {
                "flow_id": flow_id,
                "final_node": current_node_id,
                "status": "completed",
                "trace": trace,
                "actions": actions,
                "warnings": warnings
            }

        if node_type != "decision":
            raise ValueError(
                f"Nó `{current_node_id}` possui type inválido: {node_type!r}. "
                "Esperado: 'decision' ou 'terminal'."
            )

        condition = node.get("condition")
        if_true = node.get("if_true")
        if_false = node.get("if_false")
        action_if_true = node.get("action_if_true")
        action_if_false = node.get("action_if_false")
        warning = node.get("warning")

        if warning:
            warnings.append(f"{current_node_id}: {warning}")

        condition_result = evaluate_condition(condition, context)

        if condition_result:
            selected_action = action_if_true
            next_node_id = if_true
            branch = "true"
        else:
            selected_action = action_if_false
            next_node_id = if_false
            branch = "false"

        if selected_action is not None:
            actions.append({
                "node_id": current_node_id,
                "branch": branch,
                "action": selected_action
            })

        trace.append({
            "step": step_index + 1,
            "node_id": current_node_id,
            "node_type": "decision",
            "condition": condition,
            "condition_result": condition_result,
            "branch_taken": branch,
            "next_node": next_node_id,
            "description": node.get("description")
        })

        if not next_node_id:
            raise ValueError(
                f"Nó `{current_node_id}` não definiu próximo passo para o ramo `{branch}`."
            )

        current_node_id = next_node_id

    raise RuntimeError(
        f"Execução interrompida após {max_steps} passos. "
        "Possível ciclo no ruleset."
    )

In [ ]:
context = {
    "cid10": "E11",
    "glicemia": 140,
    "hemoglobina_glicada": 7.0,
    "totg": 180,
    "imc": 42,
    "antecedente_familiar_1grau_dm": False,
    "sedentarismo": True,
    "hipertensao": True,
    "doenca_arterial_coronariana": False,
    "triglicerides": 200,
    "hdl_c": 40,
    "sindrome_ovarios_policisticos": False,
    "estratificacao_risco": "alta",
    "taxa_filtracao_glomerular": 80,
    "albuminuria_mg_por_g_creatinina": 10,
    "prediabetes": False,
    "idade": 55,
}

output = execute_ruleset(ruleset, context)

In [ ]:
# output

In [ ]:
output.keys()

In [ ]:
output['trace']

***

In [ ]:
import os

print(os.getcwd())
os.chdir('../')
print(os.getcwd())

from markdown_vector_indexer.pipeline import RulesetPipeline

In [ ]:
pipeline = RulesetPipeline()

In [ ]:
tree_query = (
    "Criar árvore de decisão com enfoque no suporte à decisão médica para recomendação de medicamentos a serem prescritos ao paciente."
    "Considere que, na prática, a árvore devere permitir conclusões (i.e, recomendação de medicamentos) tomando como ponto de partida:"
    "1. o histórico de diagnósticos CID-10 do paciente;"
    "2. os medicamentos atualmente que o paciente usa;"
    "3. histórico de comorbidades do paciente;"
    "4. parâmetros antropométricos do paciente (peso, altura, IMC)."
)

In [ ]:
decisionflow = pipeline.query(tree_query) # verificar se devo melhorar nome deste (e checar se outros) métodos.

In [ ]:
decisionflow

In [ ]:
ruleset = pipeline.build_ruleset(decisionflow.flows)

In [ ]:
ruleset

In [ ]:
# import json

# os.makedirs("data/ruleset", exist_ok=True)

# path = "data/ruleset/ruleset.json"

# with open(path, "w", encoding="utf-8") as f:

#     json.dump(ruleset, f, ensure_ascii=False, indent=2)

# print(f"Ruleset salvo em: {path}")

***

Testando fluxo de ponta a ponta:

In [1]:
import os
import json

print(os.getcwd())
os.chdir('../')
print(os.getcwd())

from markdown_vector_indexer.pipeline import RulesetPipeline

/Users/drt67700/git/poc_extracao_de_regras/notebooks
/Users/drt67700/git/poc_extracao_de_regras


In [2]:
pipeline = RulesetPipeline()

In [3]:
with open("data/ruleset/ruleset_F1.json", encoding="utf-8") as f:

    ruleset = json.load(f)

In [ ]:
# context = {
#     "cid10": 'E11',
#     "glicemia": 140,
#     "hemoglobina_glicada": 7.0,
#     "totg": 180,
#     "imc": 42,
#     "antecedente_familiar_1grau_dm": False,
#     "sedentarismo": True,
#     "historico_hipertensao": True,
#     "doenca_arterial_coronariana": False,
#     "triglicerides": 200,
#     "hdl_c": 40,
#     "sindrome_ovarios_policisticos": False,
#     "estratificacao_risco": "alta",
#     "taxa_filtracao_glomerular": 80,
#     "albuminuria_mg_por_g_creatinina": 10,
#     "prediabetes": False,
#     "idade": 55,
# }

# context = {"age": 55, "sex": "male", "cid10": "E11", "estratificacao_risco": "muito_alta"}

# context = {"cid10": "e10", "imc": 42}
# context = {"cid10": "E10", "imc": 28}
# context = {"cid10": "E11", "estratificacao_de_risco": "muito_alta"}

# context = {"age": "34", "sex": "female", "cid10": "T89", "estratificacao_de_risco": "muito_alta"}
# context = {"age": "34", "sex": "female", "cid10": "E11", "estratificacao_de_risco": "muito_alta"}

# context = {"age": "47", "sex": "female", "cid10": "E11", "estratificacao_de_risco": "alta"}

# context = {"age": "61", "sex": "male", "cid10": "T89", "historico_de_saude": "[I10, G43]"}
# context = {"age": 61, "sex": "male", "cid10": "E11", "historico_de_saude": "[I10, G43]"}
# context = {"age": 61, "sex": "male", "cid10": "E11", "historico_de_saude": "[I10, G43]", "hipertensao":'sim'}

# context = {"age": "19", "sex": "male", "cid10": "E14", "taxa_de_filtração_glomerular": "100", "historico_de_saude": "[I10, G43, E11.2]"}
context = {"age": "19", "sex": "male", "cid10": "E14", "taxa_de_filtracao_glomerular": 10, "historico_de_saude": "[G43, E11.2]"}

In [ ]:
result = pipeline.solve(ruleset=ruleset, query=context)

In [ ]:
result.keys()

In [ ]:
# result['actions']
# result['warnings']
# result['trace']
result['output']

In [ ]:
result['actions']

In [ ]:
# result['warnings']
# result['output']

# ruleset['nodes']

***

In [1]:
import requests
import json

API_URL = "http://localhost:5000/rulesets/solve"

query = {"age": 55, "sex": "male", "cid10": "E11", "estratificacao_de_risco": "muito_alta",}

response = requests.post(API_URL,  json={"query_text": repr(query)}, timeout=120)

response.raise_for_status()
result = response.json()

In [3]:
result.keys()

dict_keys(['actions', 'flow_id', 'output', 'outputs', 'results', 'status', 'trace', 'warnings'])

In [3]:
# result['actions']
# result['results']#[0]
# result['status']
# result['trace']#[0]
# result['warnings']
# result['flow_id']

In [6]:
print("Status:", result.get("status"))

print("\nOutput final:")
print(result.get("output"))

print("\nOutput com mais informação:")
print(json.dumps(result.get("outputs"), ensure_ascii=False, indent=2))

print("\nRulesets executados:", len(result.get("results")))

Status: completed

Output final:
prescrever_acido_acetil_salicilico_100mg_vo_1x_dia
prescrever_metformina_500mg_xr_vo_1x_dia_com_possivel_associacao_conforme_comorbidades
associar_sglt2_ou_glp1_conforme_disponibilidade

Output com mais informação:
[
  {
    "flow_id": "F1",
    "output": "prescrever_acido_acetil_salicilico_100mg_vo_1x_dia"
  },
  {
    "flow_id": "F5",
    "output": "prescrever_metformina_500mg_xr_vo_1x_dia_com_possivel_associacao_conforme_comorbidades"
  },
  {
    "flow_id": "F6",
    "output": "associar_sglt2_ou_glp1_conforme_disponibilidade"
  }
]

Rulesets executados: 9


In [5]:
result.get("outputs")

[{'flow_id': 'F1',
  'output': 'prescrever_acido_acetil_salicilico_100mg_vo_1x_dia'},
 {'flow_id': 'F5',
  'output': 'prescrever_metformina_500mg_xr_vo_1x_dia_com_possivel_associacao_conforme_comorbidades'},
 {'flow_id': 'F6',
  'output': 'associar_sglt2_ou_glp1_conforme_disponibilidade'}]

In [13]:
result.get("actions")#[0]

[{'action': 'N2', 'branch': 'true', 'flow_id': 'F1', 'node_id': 'N1'},
 {'action': 'N4', 'branch': 'true', 'flow_id': 'F1', 'node_id': 'N2'},
 {'action': 'N2', 'branch': 'true', 'flow_id': 'F2', 'node_id': 'N1'},
 {'action': 'N5', 'branch': 'false', 'flow_id': 'F2', 'node_id': 'N2'},
 {'action': 'N2', 'branch': 'true', 'flow_id': 'F3', 'node_id': 'N1'},
 {'action': 'N5', 'branch': 'false', 'flow_id': 'F3', 'node_id': 'N2'},
 {'action': 'N3', 'branch': 'false', 'flow_id': 'F4', 'node_id': 'N1'},
 {'action': 'N2', 'branch': 'true', 'flow_id': 'F5', 'node_id': 'N1'},
 {'action': 'N4', 'branch': 'true', 'flow_id': 'F5', 'node_id': 'N2'},
 {'action': 'N2', 'branch': 'true', 'flow_id': 'F6', 'node_id': 'N1'},
 {'action': 'N5', 'branch': 'false', 'flow_id': 'F6', 'node_id': 'N2'},
 {'action': 'N6', 'branch': 'true', 'flow_id': 'F6', 'node_id': 'N5'},
 {'action': 'N2', 'branch': 'true', 'flow_id': 'F7', 'node_id': 'N1'},
 {'action': 'N5', 'branch': 'false', 'flow_id': 'F7', 'node_id': 'N2'},
 